In [1]:
"""
M3 Pandas 進階：merge / groupby / RFM — 課後作業
=================================================
情境：你已經有清理好的訂單資料，現在要合併客戶和商品表，
做出有商業價值的分析。

資料路徑：
  - datasets/ecommerce/orders_clean.csv
  - datasets/ecommerce/customers.csv
  - datasets/ecommerce/products.csv
"""

'\nM3 Pandas 進階：merge / groupby / RFM — 課後作業\n=================================================\n情境：你已經有清理好的訂單資料，現在要合併客戶和商品表，\n做出有商業價值的分析。\n\n資料路徑：\n  - datasets/ecommerce/orders_clean.csv\n  - datasets/ecommerce/customers.csv\n  - datasets/ecommerce/products.csv\n'

In [2]:
import pandas as pd


# ============================================================
# 🟢 送分題（每題 10 分，共 30 分）
# ============================================================

In [3]:
#def green_load_and_merge():

"""
    讀取三張表，合併成一張完整的 DataFrame 並回傳
    - orders_clean.csv LEFT JOIN customers.csv ON customer_id
    - 再 LEFT JOIN products.csv ON product_id
    提示：pd.merge(how='left')
    """
    # TODO: 你的程式碼

DATA = '../datasets/ecommerce'
orders    = pd.read_csv(f'{DATA}/orders_clean.csv', parse_dates=['order_date'])
customers = pd.read_csv(f'{DATA}/customers.csv')
products  = pd.read_csv(f'{DATA}/products.csv')

df = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print('分析基底:', df.shape)
df[['order_id','customer_name','region','vip_level','product_name','category','amount']].head()


分析基底: (188, 14)


,order_id,customer_name,region,vip_level,product_name,category,amount
0,5064,Victor Lin,North,Gold,Dumbbell 5kg,Sports,8600.0
1,5023,Zoe Huang,South,Platinum,Throw Pillow,Home,1355.0
2,5123,Mia Huang,North,Platinum,Cotton T-Shirt,Clothing,3538.0
3,5118,Emma Liu,West,Bronze,Water Bottle,Sports,1618.0
4,5161,Quinn Chen,East,Silver,Coffee Mug,Home,1846.0


In [4]:
def green_row_count(df):
    """回傳 DataFrame 的列數 (int)"""
    # TODO: 你的程式碼
rows = len(df)
print(rows)

188


In [5]:
def green_column_list(df):
    """回傳 DataFrame 的所有欄位名稱 (list)"""
    # TODO: 你的程式碼
df.columns

Index(['order_id', 'customer_id', 'product_id', 'qty', 'order_date', 'amount',
       'customer_name', 'region', 'signup_date', 'vip_level', 'product_name',
       'category', 'unit_price', 'stock_qty'],
      dtype='str')

# ============================================================
# 🟡 核心題（每題 15 分，共 45 分）
# ============================================================

In [6]:
def yellow_top_category(df):
    """
    哪個商品類別 (category) 的總營收最高？
    回傳該類別名稱 (str)
    提示：groupby('category')['amount'].sum()
    """
    # TODO: 你的程式碼
cat_rev = df.groupby('category')['amount'].sum().sort_values(ascending=False)
print('🏆 商品營收排名:')
print(cat_rev)


🏆 商品營收排名:
category
Books          182244.0
Sports         176315.0
Clothing       133841.0
Electronics    100235.0
Home            93753.0
Name: amount, dtype: float64


In [7]:
def yellow_gold_vip_stats(df):
    """
    Gold VIP 客戶總共下了幾張訂單？總金額多少？
    回傳 tuple: (訂單數 int, 總金額 float)
    提示：df[df['vip_level'] == 'Gold']
    """
    # TODO: 你的程式碼
#篩選出gold 客戶資料
vip_gold = df[df['vip_level'] == 'Gold']
#vip gold 訂單數量
gold_rows = len(vip_gold)
print(gold_rows) #79

#總金額
vip_gold_total_rev = vip_gold['amount'].sum()
print(vip_gold_total_rev)

79
285982.0


In [8]:
def yellow_region_avg_amount(df):
    """
    計算每個地區 (region) 的平均訂單金額
    回傳 Series（index=region, values=平均金額）
    提示：groupby('region')['amount'].mean()
    """
    # TODO: 你的程式碼
df
avg_region_rev = df.groupby('region')['amount'].mean()
print(avg_region_rev)

region
East     3318.000000
North    3738.044776
South    3786.152778
West     3340.848485
Name: amount, dtype: float64


# ============================================================
# 🔴 挑戰題（25 分）
# ============================================================

In [22]:
def red_rfm_top5(df):
    """
    RFM 分析：找出最有價值的前 5 位客戶

    計算每位客戶的：
    - R (Recency)：最近一次下單日期
    - F (Frequency)：訂單總數
    - M (Monetary)：消費總金額

    回傳 DataFrame：
    - 欄位：customer_id, customer_name, R, F, M
    - 按 M 由大到小排序
    - 只取前 5 筆

    提示：groupby('customer_id').agg(...)
    """
    # TODO: 你的程式碼
import pandas as pd
import time 
from datetime import datetime #常與dataframe 搭配使用
    #現在時間
now = time.strftime("%Y-%m-%d", time.localtime())
now = datetime.now()
print(now)
    #從現在到orderdate的距離
df['date_diff'] = (now-df["order_date"]).dt.days

#recency : 最近一次下單:取天數最少的
recency = df.groupby("customer_id")["date_diff"].max()

    #frequency : 購買頻率
frequency = df.groupby("customer_id")["order_id"].count()

    #Monetary : 總金額
monetary = df.groupby("customer_id")["amount"].sum() 

names = df.groupby("customer_id")["customer_name"].first()
    


df_RFM = pd.DataFrame({
    'customer_id':recency,
    'customer_name':names,
    'Recency': recency,
    'Frequency': frequency,
    'Monetary': monetary
 })

df_RFM.head()


2026-04-30 23:46:09.133630


,customer_id,customer_name,Recency,Frequency,Monetary
customer_id,,,,,
2001,367,Alice Chen,367,3,16095.0
2002,483,Bob Wang,483,7,16211.0
2003,469,Carol Huang,469,10,23007.0
2004,463,David Chen,463,7,39085.0
2005,483,Emma Liu,483,8,34917.0


In [ ]:
df

,order_id,customer_id,product_id,qty,order_date,amount,customer_name,region,signup_date,vip_level,product_name,category,unit_price,stock_qty,recency,frequency,monetary
0,5064,2022,1026,4.0,2025-03-26,8600.0,Victor Lin,North,2023-02-27,Gold,Dumbbell 5kg,Sports,2150,51,400 days 23:05:54.249586,NaN,NaN
1,5023,2026,1021,5.0,2025-01-05,1355.0,Zoe Huang,South,2023-05-16,Platinum,Throw Pillow,Home,271,150,480 days 23:05:54.249586,NaN,NaN
2,5123,2013,1013,2.0,2025-09-11,3538.0,Mia Huang,North,2023-07-17,Platinum,Cotton T-Shirt,Clothing,1769,174,231 days 23:05:54.249586,NaN,NaN
3,5118,2005,1028,1.0,2025-05-22,1618.0,Emma Liu,West,2023-05-18,Bronze,Water Bottle,Sports,1618,186,343 days 23:05:54.249586,NaN,NaN
4,5161,2017,1019,3.0,2025-08-20,1846.0,Quinn Chen,East,2023-08-11,Silver,Coffee Mug,Home,1846,274,253 days 23:05:54.249586,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,5094,2026,1019,3.0,2025-02-13,5538.0,Zoe Huang,South,2023-05-16,Platinum,Coffee Mug,Home,1846,274,441 days 23:05:54.249586,NaN,NaN
184,5041,2014,1001,5.0,2025-10-03,8135.0,Nick Huang,West,2023-09-28,Gold,Wireless Mouse,Electronics,1627,12,209 days 23:05:54.249586,NaN,NaN
185,5157,2005,1026,5.0,2025-01-02,10750.0,Emma Liu,West,2023-05-18,Bronze,Dumbbell 5kg,Sports,2150,51,483 days 23:05:54.249586,NaN,NaN
186,5134,2015,1012,5.0,2025-06-03,9580.0,Olivia Huang,North,2023-12-15,Bronze,Clean Code,Books,1916,81,331 days 23:05:54.249586,NaN,NaN
